# EA2 - Actividad 2.1: Ingesta de Datos

## Objetivos
- Dominar la lectura de multiples formatos de datos (CSV, JSON, Parquet, Excel)
- Comprender la diferencia entre schema inferido y schema manual
- Manejar opciones avanzadas de lectura (delimitador, encoding, nulls)
- Conocer los modos de manejo de datos corruptos
- Escribir y leer datos en formato Parquet

## Conceptos Clave

### Formatos de Datos en Big Data

| Formato | Tipo | Ventajas | Desventajas |
|---------|------|----------|-------------|
| **CSV** | Texto | Universal, legible, simple | Lento, sin tipos, sin compresion |
| **JSON** | Texto | Flexible, soporta anidamiento | Verboso, lento para grandes volumenes |
| **Parquet** | Binario/Columnar | Rapido, comprimido, tipado | No legible directamente |
| **ORC** | Binario/Columnar | Similar a Parquet, optimizado para Hive | Menos adoptado fuera de Hive |
| **Excel** | Binario | Familiar para usuarios de negocio | No escalable, requiere pandas |

### Schema Inferido vs Schema Manual

- **Schema inferido (`inferSchema=True`):** Spark lee una muestra de los datos para adivinar los tipos. Es comodo pero puede ser lento en archivos grandes y cometer errores (ej: confundir strings con integers).
- **Schema manual (`StructType`):** Tu defines explicitamente cada columna y su tipo. Es mas rapido y confiable para produccion.

### Modos de Manejo de Datos Corruptos

| Modo | Comportamiento |
|------|---------------|
| `PERMISSIVE` | Coloca valores null donde hay datos corruptos (default) |
| `DROPMALFORMED` | Elimina filas con datos corruptos |
| `FAILFAST` | Lanza una excepcion al encontrar datos corruptos |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("EA2_01_ingesta_datos") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Leer CSV con Schema Inferido

La forma mas simple de leer un CSV. Spark analiza los datos para inferir automaticamente los tipos de cada columna.

In [ ]:
# Lectura basica de CSV con inferencia de schema
df_flights = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# Verificar el schema inferido
print("Schema inferido automaticamente:")
df_flights.printSchema()

# Ver las primeras filas
df_flights.show(5)

# Informacion basica
print(f"Filas: {df_flights.count()}")
print(f"Columnas: {len(df_flights.columns)}")

## 2. Leer CSV con Schema Manual

Definir el schema manualmente es una **buena practica en produccion** porque:
1. Es mas rapido (Spark no necesita analizar los datos)
2. Garantiza los tipos correctos
3. Permite controlar la nulabilidad

Se usa `StructType` con una lista de `StructField(nombre, tipo, nullable)`.

In [ ]:
# Definir schema manualmente
schema_flights = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("AIRLINE", StringType(), True),
    StructField("ORIGIN_AIRPORT", StringType(), True),
    StructField("DESTINATION_AIRPORT", StringType(), True),
    StructField("DEPARTURE_DELAY", DoubleType(), True),
    StructField("ARRIVAL_DELAY", DoubleType(), True),
    StructField("DISTANCE", IntegerType(), True),
])

# Leer con schema manual
df_flights_manual = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, schema=schema_flights)

# Verificar que el schema es exactamente el que definimos
print("Schema manual:")
df_flights_manual.printSchema()

df_flights_manual.show(5)

## 3. Opciones Avanzadas de Lectura

`spark.read.csv()` acepta multiples opciones para manejar archivos con formatos especiales.

In [ ]:
# Ejemplo de opciones avanzadas de lectura CSV
# (Usando el mismo archivo como ejemplo de sintaxis)

df_opciones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("encoding", "UTF-8") \
    .option("nullValue", "NA") \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("/home/jovyan/datos/flights.csv")

print("Lectura con opciones avanzadas exitosa")
df_opciones.show(5)

**Opciones mas comunes:**

| Opcion | Descripcion | Ejemplo |
|--------|-------------|----------|
| `delimiter` | Separador de campos | `","`, `";"`, `"\t"` |
| `encoding` | Codificacion del archivo | `"UTF-8"`, `"ISO-8859-1"` |
| `nullValue` | String que representa null | `"NA"`, `"NULL"`, `""` |
| `dateFormat` | Formato de fechas | `"yyyy-MM-dd"`, `"dd/MM/yyyy"` |
| `quote` | Caracter de comillas | `'"'` |
| `escape` | Caracter de escape | `'\\'` |
| `multiLine` | Campos con saltos de linea | `"true"` |

## 4. Leer Archivos Excel

Spark no lee Excel nativamente. La estrategia es:
1. Leer con `pandas` (que si soporta Excel)
2. Convertir el DataFrame de pandas a un DataFrame de Spark

In [ ]:
# Leer Excel usando pandas como puente
import pandas as pd

df_pandas = pd.read_excel("/home/jovyan/datos/ufo_sightings.xlsx")
print(f"DataFrame pandas: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")

# Convertir a DataFrame de Spark
df_ufo = spark.createDataFrame(df_pandas)
print("\nSchema del DataFrame de Spark:")
df_ufo.printSchema()
df_ufo.show(5, truncate=50)

## 5. Escribir y Leer Parquet

**Parquet** es el formato preferido en Big Data porque:
- Es **columnar**: permite leer solo las columnas necesarias
- Tiene **compresion** integrada (snappy, gzip, etc.)
- Preserva los **tipos de datos** en el schema
- Es mucho mas **rapido** de leer que CSV

In [ ]:
# Escribir DataFrame a formato Parquet
df_flights_manual.write.mode("overwrite").parquet("/home/jovyan/datos/flights_parquet")

print("Archivo Parquet escrito exitosamente")

In [ ]:
# Leer desde Parquet
df_parquet = spark.read.parquet("/home/jovyan/datos/flights_parquet")

# Notar que el schema se preserva automaticamente
print("Schema leido desde Parquet (sin necesidad de inferir):")
df_parquet.printSchema()

df_parquet.show(5)
print(f"Filas: {df_parquet.count()}")

## 6. Modos de Manejo de Datos Corruptos

Cuando los datos no coinciden con el schema esperado, Spark ofrece tres estrategias.

In [ ]:
# Modo PERMISSIVE (default): pone null en columnas corruptas
df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema_flights) \
    .csv("/home/jovyan/datos/flights.csv")

print("Modo PERMISSIVE: filas leidas =", df_permissive.count())

In [ ]:
# Modo DROPMALFORMED: elimina filas corruptas
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(schema_flights) \
    .csv("/home/jovyan/datos/flights.csv")

print("Modo DROPMALFORMED: filas leidas =", df_drop.count())

In [ ]:
# Modo FAILFAST: lanza excepcion si encuentra datos corruptos
# Descomentar para probar (puede lanzar error):
# df_fail = spark.read \
#     .option("header", "true") \
#     .option("mode", "FAILFAST") \
#     .schema(schema_flights) \
#     .csv("/home/jovyan/datos/flights.csv")

print("Modo FAILFAST: lanza excepcion al encontrar datos corruptos")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Leer netflix_titles.csv con schema manual
# =============================================================
# Definir schema manual con StructType

schema_netflix = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True),
])

# Leer el archivo con schema manual
df_netflix = spark.read.csv(
    "/home/jovyan/datos/netflix_titles.csv",
    header=True,
    schema=schema_netflix
)

# 1. Mostrar schema
print("Schema del DataFrame:")
df_netflix.printSchema()

# 2. Mostrar primeras 5 filas
print("Primeras 5 filas:")
df_netflix.show(5, truncate=50)

# 3. Conteo total
print(f"Total de filas: {df_netflix.count()}")

> **Nota docente:** Se usa StringType para la mayoria de las columnas porque muchos campos
> de texto contienen comas internas (cast, listed_in, description), lo que podria causar
> problemas de parsing. El unico campo numerico claro es `release_year` (IntegerType).
> Mantener `date_added` como StringType es intencional: se parseara en la fase de
> transformacion (notebook 02). Un error comun de los estudiantes es intentar usar
> DateType aqui, pero el formato "Month dd, yyyy" requiere `to_date()` con formato explicito.

In [ ]:
# =============================================================
# EJERCICIO 2: Explorar schema inferido de cartas_magic.csv
# =============================================================

# Leer con inferSchema=True
df_magic = spark.read.csv(
    "/home/jovyan/datos/cartas_magic.csv",
    header=True,
    inferSchema=True
)

# 1. Mostrar schema inferido
print("Schema inferido:")
df_magic.printSchema()

# 2. Mostrar primeras 10 filas
print("Primeras 10 filas:")
df_magic.show(10, truncate=40)

# 3. Analisis de tipos incorrectos:
# - La columna 'power' y 'toughness' pueden contener valores como '*'
#   (criaturas con poder/resistencia variable), por lo que Spark las infiere
#   como string, pero conceptualmente son numericas.
# - La columna 'loyalty' (para planeswalkers) puede tener nulls y valores
#   mixtos que confunden al inferidor.
# - Columnas como 'colors' o 'colorIdentity' son listas representadas como
#   strings (ej: "[W, U]"), no arrays reales.
# - 'convertedManaCost' podria inferirse como double cuando deberia ser int.

print("\nAnalisis de tipos potencialmente incorrectos:")
print("  - power/toughness: inferidos como string por valores como '*'")
print("  - colors/colorIdentity: inferidos como string, deberian ser arrays")
print("  - Campos de fecha (si existen): inferidos como string en lugar de DateType")

> **Nota docente:** El objetivo de este ejercicio es que el estudiante entienda que
> `inferSchema` no siempre acierta. Los casos mas comunes de error son:
> 1. Columnas que mezclan numeros y texto (power/toughness con `*`)
> 2. Columnas de fecha que se leen como string
> 3. Columnas con listas serializadas como texto
> 
> En produccion, siempre se recomienda definir el schema manualmente para evitar
> sorpresas. El estudiante deberia mencionar al menos 2 de estos problemas.

In [ ]:
# =============================================================
# EJERCICIO 3: Convertir CSV a Parquet y comparar tamanos
# =============================================================
import os

# 1. Leer flights.csv
df_flights_ej = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# 2. Escribir en formato Parquet
df_flights_ej.write.mode("overwrite").parquet("/home/jovyan/datos/flights_ejercicio_parquet")
print("Archivo Parquet escrito exitosamente")

# 3. Comparar tamanos
# Tamano del CSV original
tamano_csv = os.path.getsize("/home/jovyan/datos/flights.csv")
print(f"\nTamano CSV: {tamano_csv:,} bytes ({tamano_csv / 1024 / 1024:.2f} MB)")

# Tamano del directorio Parquet (sumar todos los archivos .parquet)
directorio_parquet = "/home/jovyan/datos/flights_ejercicio_parquet"
tamano_parquet = 0
for archivo in os.listdir(directorio_parquet):
    ruta_completa = os.path.join(directorio_parquet, archivo)
    if os.path.isfile(ruta_completa):
        tamano_parquet += os.path.getsize(ruta_completa)

print(f"Tamano Parquet: {tamano_parquet:,} bytes ({tamano_parquet / 1024 / 1024:.2f} MB)")

# 4. Calcular porcentaje de compresion
compresion = (1 - tamano_parquet / tamano_csv) * 100
print(f"\nCompresion: {compresion:.1f}%")
print(f"Parquet es {tamano_csv / tamano_parquet:.1f}x mas pequeno que CSV")

> **Nota docente:** El ratio de compresion tipico de CSV a Parquet (con Snappy,
> compresion por defecto) es entre 3x y 10x, dependiendo de los datos. Los factores
> que afectan la compresion son:
> - Columnas con muchos valores repetidos comprimen mejor (ej: AIRLINE)
> - Datos numericos comprimen mejor que texto largo
> - Parquet almacena por columna, lo que permite compresion mas eficiente
>
> Un error comun es no incluir los archivos de metadata (como `_SUCCESS`) al calcular
> el tamano. Se recomienda sumar solo los `.parquet` pero en la practica la diferencia
> es minima porque los archivos de metadata son muy pequenos.

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Escribe una funcion `resumen_csv(path)` que reciba la ruta de cualquier archivo CSV y muestre un resumen completo del archivo.

In [ ]:
# =============================================================
# DESAFIO: Funcion de resumen automatico de CSV
# =============================================================

def resumen_csv(spark, path):
    """Muestra un resumen completo de un archivo CSV.
    
    Parametros:
        spark: SparkSession activa
        path: Ruta al archivo CSV
    """
    # Leer el CSV con schema inferido
    df = spark.read.csv(path, header=True, inferSchema=True)
    
    # 1. Numero total de filas
    total_filas = df.count()
    print("=" * 60)
    print(f"RESUMEN DEL ARCHIVO: {path}")
    print("=" * 60)
    print(f"\n1. Total de filas: {total_filas:,}")
    
    # 2. Numero de columnas
    print(f"2. Total de columnas: {len(df.columns)}")
    
    # 3. Nombre y tipo de cada columna
    print(f"\n3. Columnas y tipos:")
    print(f"   {'Columna':<30s} {'Tipo':<15s}")
    print(f"   {'-'*30} {'-'*15}")
    for campo in df.schema.fields:
        print(f"   {campo.name:<30s} {str(campo.dataType):<15s}")
    
    # 4. Cantidad de valores null por columna
    print(f"\n4. Valores nulos por columna:")
    nulls_df = df.select(
        [F.count(F.when(F.isnull(c), c)).alias(c) for c in df.columns]
    ).collect()[0]
    
    print(f"   {'Columna':<30s} {'Nulls':<10s} {'% Nulls':<10s}")
    print(f"   {'-'*30} {'-'*10} {'-'*10}")
    for col_name in df.columns:
        null_count = nulls_df[col_name]
        pct = (null_count / total_filas * 100) if total_filas > 0 else 0
        print(f"   {col_name:<30s} {null_count:<10d} {pct:<.2f}%")
    
    # 5. Valores unicos por columna (opcional)
    print(f"\n5. Valores unicos por columna:")
    print(f"   {'Columna':<30s} {'Unicos':<10s}")
    print(f"   {'-'*30} {'-'*10}")
    for col_name in df.columns:
        n_unicos = df.select(col_name).distinct().count()
        print(f"   {col_name:<30s} {n_unicos:<10d}")
    
    print("\n" + "=" * 60)
    return df


# Probar con flights.csv
print(">>> Probando con flights.csv")
df1 = resumen_csv(spark, "/home/jovyan/datos/flights.csv")

In [ ]:
# Probar con netflix_titles.csv
print(">>> Probando con netflix_titles.csv")
df2 = resumen_csv(spark, "/home/jovyan/datos/netflix_titles.csv")

> **Nota docente:** Esta funcion es un patron muy util en la practica. Puntos clave:
> - Se usa `F.count(F.when(F.isnull(c), c))` para contar nulls de forma eficiente
>   en una sola pasada sobre los datos.
> - El conteo de valores unicos (`distinct().count()`) puede ser costoso en datasets
>   grandes. En produccion se usaria `approx_count_distinct()` para una estimacion
>   mas rapida.
> - La funcion retorna el DataFrame para que se pueda seguir trabajando con el.
> - Un error comun es no manejar el caso de `total_filas = 0` al calcular porcentajes.

---
## Resumen

En esta actividad aprendimos:

1. **Lectura de CSV:** `spark.read.csv(path, header=True, inferSchema=True)` para lectura rapida
2. **Schema manual:** `StructType` + `StructField` para definir tipos explicitamente (recomendado en produccion)
3. **Opciones avanzadas:** delimiter, encoding, nullValue, dateFormat para archivos con formatos especiales
4. **Lectura de Excel:** Usar pandas como puente: `spark.createDataFrame(pd.read_excel(path))`
5. **Parquet:** Formato columnar comprimido, ideal para Big Data: `.write.parquet()` y `.read.parquet()`
6. **Datos corruptos:** Tres modos de manejo: PERMISSIVE (default), DROPMALFORMED, FAILFAST

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")